# v0.29.0 — Multiple databases

## 1. Intro

Since v0.29.0 one process can work with **several databases at once**. Beside the default connection
(`set_connection()`), `add_connection()` registers **named connections** — another database on the same server,
another server, or the embedded engine — and each model reads and writes through its own: `SurrealConfigDict(connection=…)`
routes a model, and `using(name)` redirects a block of code. Each connection keeps its own clients, session, live
streams, transactions and cached reads, and decides on its own server line, so a SurrealDB 2.x server and a 3.x server
work side by side in one process.

This notebook uses three connections: the server's `examples` database (the default connection), a second database on
the same server (`analytics`), and the in-process embedded engine (`scratch`, `mem://`). It runs top to bottom on
SurrealDB 2.x and 3.x; only the cell that prints the server line and the transaction section (§6) differ.

## 2. Connect

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket.
`add_connection()` takes the settings of `set_connection()`, checks them before registering anything, and opens
nothing: each connection opens its own client on first use. `list_connections()` returns the registered names,
`'default'` first.

In [1]:
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
SERVER_URL = f"ws://{HOST}:{PORT}/rpc"

SurrealDBConnectionManager.set_connection(  # the default connection
    url=SERVER_URL,
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
SurrealDBConnectionManager.add_connection(  # another database on the same server
    "analytics",
    url=SERVER_URL,
    user="root",
    password="root",
    namespace="examples",
    database="examples_analytics",
)
SurrealDBConnectionManager.add_connection(  # the embedded engine, in this process
    "scratch",
    url="mem://",
    namespace="nb",
    database="nb",
)
print("Connections:", SurrealDBConnectionManager.list_connections())

Connections: ['default', 'analytics', 'scratch']


`get_config(name)` returns a connection's settings as a frozen `ConnectionConfig`, the password kept out of its `repr`
(here the embedded one: the server's URL would show the port).

In [2]:
print(SurrealDBConnectionManager.get_config("scratch"))
analytics = SurrealDBConnectionManager.get_config("analytics")
print("analytics:", analytics.namespace, "/", analytics.database)

client = await SurrealDBConnectionManager.get_client()  # the default connection's SDK client
version = str(await client.version())  # "surrealdb-<major>.<minor>.<patch>"
line = int(version.removeprefix("surrealdb-").split(".")[0])
print(f"Server: SurrealDB {line}.x")

ConnectionConfig(url='mem://', user=None, namespace='nb', database='nb')
analytics: examples / examples_analytics
Server: SurrealDB 3.x


## 3. Models

`Customer` has no `connection` key, so it uses the default connection; `PageView` is routed to `analytics` and
`Draft` to `scratch`. The key is checked when the class is defined (a non-empty string); whether the name is
registered is checked at the model's first database call, so models can be declared before `add_connection()` runs.
`get_connection_name()` tells which connection a model's calls use right now.

The notebook starts by dropping its tables in every database, so it can be re-executed in place (on SurrealDB 3.x
removing a missing table raises, hence the suppressed errors).

In [3]:
from surreal_orm_lite import BaseSurrealModel, SurrealConfigDict


class Customer(BaseSurrealModel):  # no connection key: "default"
    id: str | None = None
    name: str


class PageView(BaseSurrealModel):
    model_config = SurrealConfigDict(connection="analytics")
    id: str | None = None
    customer: str  # the customer's name: a link would be followed in the wrong database
    path: str


class Draft(BaseSurrealModel):
    model_config = SurrealConfigDict(connection="scratch")
    id: str | None = None
    title: str


DEMO_TABLES = ["Customer", "PageView", "Draft"]


async def reset() -> None:
    """Drop every table of this notebook in every database, whether or not it exists."""
    for connection in SurrealDBConnectionManager.list_connections():
        connection_client = await SurrealDBConnectionManager.get_client(connection)
        for table in DEMO_TABLES:
            with contextlib.suppress(Exception):
                await connection_client.query(f"REMOVE TABLE {table};", {})


await reset()
for model in (Customer, PageView, Draft):
    print(f"{model.__name__:<9} runs on {model.get_connection_name()!r}")

Customer  runs on 'default'
PageView  runs on 'analytics'
Draft     runs on 'scratch'


## 4. Writes and reads, each in its own database

Every database call of a model goes to its connection: the `save()` calls and the QuerySets below each run in their
model's database, with nothing to pass.

In [4]:
for key, name in [("ada", "Ada"), ("bob", "Bob"), ("cy", "Cy")]:
    await Customer(id=key, name=name).save()
for key, customer, path in [("v1", "Ada", "/pricing"), ("v2", "Bob", "/docs"), ("v3", "Ada", "/docs")]:
    await PageView(id=key, customer=customer, path=path).save()
await Draft(id="d1", title="Spring launch").save()

print("Customer.objects().count():", await Customer.objects().count())
print("PageView.objects().count():", await PageView.objects().count())
print("Draft.objects().count():   ", await Draft.objects().count())
ada_views = await PageView.objects().filter(customer="Ada").order_by("id").exec()
print("Ada's page views:", [view.path for view in ada_views])

Customer.objects().count(): 3
PageView.objects().count(): 3
Draft.objects().count():    1
Ada's page views: ['/pricing', '/docs']


`get_client(name)` returns a connection's own SDK client. Counting each table with raw SurrealQL in every database
shows where the rows landed: each table holds rows in its model's database only (on SurrealDB 3.x reading a table
that was never created raises, so `count_in()` counts it 0).

In [5]:
async def count_in(connection: str, table: str) -> int:
    """Count a table's rows with the connection's own SDK client; 0 when the table does not exist there."""
    connection_client = await SurrealDBConnectionManager.get_client(connection)
    try:
        rows = await connection_client.query(f"SELECT count() AS n FROM {table} GROUP ALL;", {})
    except Exception:  # SurrealDB 3.x: no such table in this database
        return 0
    return rows[0]["n"] if rows else 0


print(f"{'':<10}" + "".join(f"{table:>10}" for table in DEMO_TABLES))
for connection in SurrealDBConnectionManager.list_connections():
    counts = [await count_in(connection, table) for table in DEMO_TABLES]
    print(f"{connection:<10}" + "".join(f"{n:>10}" for n in counts))

            Customer  PageView     Draft
default            3         0         0
analytics          0         3         0
scratch            0         0         1


## 5. `using()`: redirect a block of code

`using(name)` runs a block against a named connection: inside it, every model — whatever its `connection` key — and
every manager method that talks to a database use that connection. `Customer` has no rows in `analytics`, so the same
count reads 0 inside the block and 3 outside. The connection is resolved at every database call: a QuerySet built
outside the block and run inside it uses the block's connection. Blocks nest, and the outer connection is back after
the inner block, on an exception too. `get_active_connection_name()` returns the block's name, `None` outside any
block.

In [6]:
customers = Customer.objects()  # built outside any block

async with SurrealDBConnectionManager.using("analytics"):
    print("using('analytics'):        active", SurrealDBConnectionManager.get_active_connection_name(), end="")
    print(f" | Customer runs on {Customer.get_connection_name()!r} | count: {await customers.count()}")
    async with SurrealDBConnectionManager.using("scratch"):
        print(f"  nested using('scratch'):  Customer runs on {Customer.get_connection_name()!r}", end="")
        print(f" | count: {await Customer.objects().count()}")
    print(f"  back in the outer block:  Customer runs on {Customer.get_connection_name()!r}")
print("outside any block:         active", SurrealDBConnectionManager.get_active_connection_name(), end="")
print(f" | Customer runs on {Customer.get_connection_name()!r} | count: {await customers.count()}")

using('analytics'):        active analytics | Customer runs on 'analytics' | count: 0
  nested using('scratch'):  Customer runs on 'scratch' | count: 0
  back in the outer block:  Customer runs on 'analytics'
outside any block:         active None | Customer runs on 'default' | count: 3


The name lives in a `contextvars.ContextVar`, so `using()` is async-safe: concurrent tasks each see their own block.

In [7]:
import asyncio


async def count_customers(connection: str) -> tuple[str, int]:
    async with SurrealDBConnectionManager.using(connection):
        await asyncio.sleep(0)  # let the other task run inside its own block
        return connection, await Customer.objects().count()


print(await asyncio.gather(count_customers("default"), count_customers("analytics"), count_customers("scratch")))

[('default', 3), ('analytics', 0), ('scratch', 0)]


## 6. Transactions, one connection each

`transaction(name)`, or `transaction()` inside `using(name)`, opens a transaction on that connection, with the
strategy of its own server: **interactive** on a SurrealDB 3.x server over WebSocket, **buffered** on 2.x, over HTTP
and on the embedded engine. So on a 3.x server, `default` and `analytics` are interactive while `scratch` buffers; on
2.x every connection buffers. The `begin()` probe tells which line the server runs. Transactions of two connections
are independent: nothing commits across connections.

In [8]:
async def native_tx_supported(connection: str) -> bool:
    """True if the connection exposes the SDK's native transaction RPC (a SurrealDB 3.x server)."""
    connection_client = await SurrealDBConnectionManager.get_client(connection)
    try:
        txn = await connection_client.begin()
    except Exception:
        return False
    with contextlib.suppress(Exception):
        await connection_client.cancel(txn)
    return True


interactive = await native_tx_supported("default")
print("Interactive transactions on the server (SurrealDB 3.x):", interactive)
for connection in SurrealDBConnectionManager.list_connections():
    async with SurrealDBConnectionManager.transaction(connection) as tx:
        strategy = type(tx).__name__
    print(f"  transaction({connection!r}):".ljust(30), strategy)

Interactive transactions on the server (SurrealDB 3.x): True
  transaction('default'):      InteractiveTransaction
  transaction('analytics'):    InteractiveTransaction
  transaction('scratch'):      BufferedTransaction


A transaction on `analytics` writes a page view. On SurrealDB 3.x a read inside it (`objects(tx=tx)`) sees the
uncommitted write; on 2.x the write is buffered until the commit and a read inside would raise.

In [9]:
async with SurrealDBConnectionManager.transaction("analytics") as tx:
    await PageView(id="v4", customer="Cy", path="/pricing").save(tx=tx)
    if interactive:
        print("inside the transaction, PageView count:", await PageView.objects(tx=tx).count())
    else:
        print("SurrealDB 2.x: the write is buffered until the commit; a read inside the transaction would raise.")
print("after the commit, PageView count:      ", await PageView.objects().count())

inside the transaction, PageView count: 4
after the commit, PageView count:       4


## 7. A session per connection

Each connection keeps its own session: `signin()`, `authenticate()`, `invalidate()`, `info()` and `clear_session()`
act on the current connection, so sign in inside `using(name)`. `is_authenticated()` tells whether the connection
holds a token signed in through the ORM, which it replays on reconnect — on its own connection only: true on
`analytics` after the sign-in, still false on the default connection.

In [10]:
async with SurrealDBConnectionManager.using("analytics"):
    await SurrealDBConnectionManager.signin(username="root", password="root")
    print("analytics: signed in through the ORM:", SurrealDBConnectionManager.is_authenticated())
    print("analytics: PageView count:           ", await PageView.objects().count())
print("default:   signed in through the ORM:", SurrealDBConnectionManager.is_authenticated())

analytics: signed in through the ORM: True
analytics: PageView count:            4
default:   signed in through the ORM: False


## 8. Guards: one statement, one transaction, one database

SurrealDB runs a statement, and a transaction, in one database. So the ORM refuses, **before anything is sent**, an
operation that would mix two connections: a subquery of another connection's model (SurrealDB would read it in the
outer statement's database), a model of another connection in a transaction, and relations or links between models
of two connections. Each raises `SurrealDbError`.

A subquery first: the page views live in `analytics`, the customers in the default database.

In [11]:
from surreal_orm_lite import SurrealDbError

pricing_views = PageView.objects().filter(path="/pricing").select("customer")
try:
    await Customer.objects().filter(name__in=pricing_views).exec()
except SurrealDbError as error:
    print(f"{type(error).__name__}: {error}")

SurrealDbError: PageView (connection 'analytics') cannot be read in a subquery of Customer (connection 'default'): SurrealDB runs a subquery in the database of the statement around it. Read PageView first through its own connection and filter on its values, or route both models to one connection.


What works instead: read the other connection's rows through their own connection, then filter on their values.

In [12]:
viewers = sorted({view.customer for view in await PageView.objects().filter(path="/pricing").exec()})
buyers = await Customer.objects().filter(name__in=viewers).order_by("name").exec()
print("customers who viewed /pricing:", [customer.name for customer in buyers])

customers who viewed /pricing: ['Ada', 'Cy']


A transaction is one connection's: a model of another connection given its `tx=` raises, before any request. The
error leaves the block, so the transaction is cancelled and nothing is committed, Dee included.

In [13]:
try:
    async with SurrealDBConnectionManager.transaction() as tx:  # the default connection
        await Customer(id="dee", name="Dee").save(tx=tx)
        await PageView(id="v5", customer="Dee", path="/signup").save(tx=tx)  # PageView runs on "analytics"
except SurrealDbError as error:
    print(f"{type(error).__name__}: {error}")
print("Customer count:", await Customer.objects().count(), "| PageView count:", await PageView.objects().count())

SurrealDbError: PageView.save() cannot run in this transaction: the transaction belongs to connection 'default' and PageView runs on connection 'analytics'. A transaction is one connection's (one database): open it on 'analytics' — SurrealDBConnectionManager.transaction('analytics') — or run the block inside using('default').
Customer count: 3 | PageView count: 4


## 9. An on-disk embedded store, one connection at a time

Two in-process engines on one on-disk store (`surrealkv://`, `file://`) would miss each other's writes, so a store is
open under one connection name at a time in a process: a second name raises `SurrealDbConnectionError` until the
first one closes it (`close_connection()` or `remove_connection()`). Two `mem://` connections are two separate
databases. The store below lives in a temporary directory, whose path is hidden in the output. Once `notes` is
removed, `notes_copy` opens the store and reads what `notes` wrote; inside `using("notes_copy")` the `Note` model runs
there, whatever its `connection` key.

In [14]:
import tempfile

from surreal_orm_lite import SurrealDbConnectionError


class Note(BaseSurrealModel):
    model_config = SurrealConfigDict(connection="notes")
    id: str | None = None
    text: str


with tempfile.TemporaryDirectory() as directory:
    store = f"surrealkv://{directory}/notes"
    SurrealDBConnectionManager.add_connection("notes", url=store, namespace="nb", database="nb")
    SurrealDBConnectionManager.add_connection("notes_copy", url=store, namespace="nb", database="nb")

    await Note(id="n1", text="Kept on disk").save()  # opens the store for "notes"
    try:
        await SurrealDBConnectionManager.get_client("notes_copy")
    except SurrealDbConnectionError as error:
        print(f"{type(error).__name__}: {str(error).replace(directory, '<tmp>')}")

    await SurrealDBConnectionManager.remove_connection("notes")  # closes the store
    async with SurrealDBConnectionManager.using("notes_copy"):
        print("read through 'notes_copy':", [note.text for note in await Note.objects().exec()])
    await SurrealDBConnectionManager.remove_connection("notes_copy")

print("Connections:", SurrealDBConnectionManager.list_connections())

SurrealDbConnectionError: Connection 'notes_copy' cannot open 'surrealkv://<tmp>/notes': connection 'notes' already has that store open in this process, and two in-process engines on one store miss each other's writes. Use one connection for the store, or close 'notes' first.
read through 'notes_copy': ['Kept on disk']
Connections: ['default', 'analytics', 'scratch']


## 10. Cleanup

Drop the demo tables and the `analytics` database, unregister the named connections — `remove_connection()` closes
what each one holds, and the `scratch` database, in memory, goes with it — then close every connection with
`close_all_connections()`.

In [15]:
await reset()
await SurrealDBConnectionManager.remove_connection("analytics")
await SurrealDBConnectionManager.remove_connection("scratch")
client = await SurrealDBConnectionManager.get_client()
with contextlib.suppress(Exception):
    await client.query("REMOVE DATABASE examples_analytics;", {})
print("Connections:", SurrealDBConnectionManager.list_connections())
await SurrealDBConnectionManager.close_all_connections()
print("Done")

Connections: ['default']
Done
